# Full grid on an A100**Turkish -> Azerbaijani transfer mechanisms** - DLE-AI-202 Track 1.Runs the frozen 114-run grid: 228,000 Azerbaijani optimizer steps, 30 sharedTurkish checkpoints (33,780 steps), ~295,000 step-equivalents includingvalidation.## Order of operations, and why| Step | Why it comes here ||---|---|| 1-5 Verify | Pinned stack, frozen-config hashes, tests, queue shape || 6 Data + transplant | Everything downstream loads these artifacts || 7 **Benchmark** | Measures real s/step **and real per-process VRAM** || 8 **Tranche A** | ~3 h; settles throughput, escape behaviour and end-to-end integrity || 9 Decision | Apply the pre-registered rule *before* committing the window || 10 Phase 1 | Builds all 30 Turkish checkpoints **serially**, then seals the cache || 11 Phase 2 | The rest of the grid, optionally parallel, cache **read-only** || 12 Analysis | Tables and figures, regenerated from `results/` alone |**Do not skip 7-9 to save an hour.** No run had ever completed under the frozenconfiguration before this pass; Tranche A is the integration test as well asthe first measurement, and its ~3 h protects the remaining ~45.## The two constraints that shape everything- **~2 days** of scheduled window.- **~10-12 GB peak GPU memory *per team*** - not per process, and not the  card's capacity. Exceeding it is an automatic deduction, on a machine shared  with other teams.`run.vram_ceiling_gb` enforces the second: the launcher caps `parallel_streams`to what actually fits and logs the reduction. **A projected ~5.5 GB per processmeans 2 streams reach ~11 GB and 3 reach ~16.5 GB - three streams are notreachable inside the allocation for this model.** Step 7 measures the realfigure; write it back before Phase 2. See `docs/COMPUTE_ESTIMATES.md` section 2.## Two-phase executionThe 30 Turkish checkpoints are shared across 114 runs. Two concurrent streamsbuilding the same uncached checkpoint would both write the same path, producinga weights file that still **loads** but holds interleaved values - silentlywrong results with no error anywhere. Rather than mitigate that with a lock,Phase 1 builds everything serially and seals a manifest; Phase 2 verifies themanifest and opens the cache read-only, refusing to start otherwise. The racebecomes impossible rather than unlikely.

## 1-5 - Setup and verification

In [ ]:
# ---------------------------------------------------------------- 1. the repo
# Three ways to get the code in. Set ONE of these and run.
REPO_URL   = ""          # e.g. "https://github.com/<user>/az-tokenizer-transfer.git"
DRIVE_ZIP  = ""          # e.g. "/content/drive/MyDrive/az-tokenizer-transfer.zip"
UPLOAD_ZIP = True        # otherwise: prompt for a local .zip upload

import os, shutil, subprocess, sys, zipfile
from pathlib import Path

WORK = Path("/content/project")

if REPO_URL:
    if WORK.exists(): shutil.rmtree(WORK)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK)], check=True)
elif DRIVE_ZIP:
    WORK.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DRIVE_ZIP) as z: z.extractall(WORK)
elif UPLOAD_ZIP:
    from google.colab import files
    up = files.upload()
    WORK.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(next(iter(up))) as z: z.extractall(WORK)

# The zip may contain a single top-level folder; descend into it.
if not (WORK / "configs" / "experiment.yaml").exists():
    inner = [p for p in WORK.iterdir() if p.is_dir() and (p / "configs").exists()]
    if len(inner) == 1: WORK = inner[0]

assert (WORK / "configs" / "experiment.yaml").exists(), f"repo not found under {WORK}"
os.chdir(WORK); sys.path.insert(0, str(WORK))
print("project root:", WORK)
print(sorted(p.name for p in WORK.iterdir())[:14])

In [ ]:
# ---------------------------------------------------------------- 2. pinned environment
# requirements.lock is the exact stack the results were produced on. Version
# drift between machines is precisely what invalidates a cross-machine
# comparison, so we install the lock rather than letting pip resolve fresh.
#
# This REPLACES Colab's preinstalled torch/numpy and therefore REQUIRES a
# runtime restart afterwards. Expect 5-9 minutes.
!pip install -q -r requirements.lock --extra-index-url https://download.pytorch.org/whl/cu121

print("\n" + "=" * 66)
print("RESTART THE RUNTIME NOW (Runtime -> Restart session), then run the")
print("next cell. Do NOT re-run the cells above after restarting.")
print("=" * 66)

In [ ]:
# ---------------------------------------------------------------- 3. after the restart
import os, sys
from pathlib import Path
WORK = Path("/content/project")
if not (WORK / "configs").exists():
    WORK = next(p for p in WORK.iterdir() if (p / "configs").exists())
os.chdir(WORK); sys.path.insert(0, str(WORK))

import torch, transformers, numpy
print("cwd        ", Path.cwd())
print("torch      ", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("numpy      ", numpy.__version__)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("gpu        ", p.name, f"{p.total_memory/1e9:.1f} GB")
else:
    print("*** NO GPU. Runtime -> Change runtime type -> GPU.")

In [ ]:
# ---------------------------------------------------------------- 4. persistence
# Colab sessions end without warning. `run.skip_existing` resumes from durable
# result files, so pointing results/ and the Turkish cache at Drive turns a
# dropped session from lost work into a pause.
USE_DRIVE = True

import os, shutil
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORE = Path("/content/drive/MyDrive/az-tokenizer-transfer")
    for sub in ("results", "artifacts"):
        (STORE / sub).mkdir(parents=True, exist_ok=True)
        local = Path.cwd() / sub
        if local.is_symlink(): local.unlink()
        elif local.exists():
            # Preserve anything already committed in the repo (results/*.json).
            for item in local.iterdir():
                target = STORE / sub / item.name
                if not target.exists():
                    (shutil.copytree if item.is_dir() else shutil.copy2)(item, target)
            shutil.rmtree(local)
        local.symlink_to(STORE / sub, target_is_directory=True)
    print("results/ and artifacts/ -> ", STORE)
else:
    print("Ephemeral storage: everything is lost when the session ends.")

import shutil as _sh
free = _sh.disk_usage(Path.cwd()).free / 1e9
print(f"free disk: {free:.1f} GB")

In [ ]:
# ---------------------------------------------------------------- 5. verification gate
# Nothing is launched until the frozen config matches its hash lock, the test
# suite is green, and the queue has the expected shape. Each of these has
# caught a real defect in this project.
!python -c "from src.utils import verify_config_hashes; verify_config_hashes(); print('frozen config hashes OK')"
!python -m pytest tests/ -q -m "not slow" --no-header
!python -m src.training.run_grid --config configs/experiment.yaml --dry-run | head -3

In [ ]:
# Confirm this is actually an A100 and report the slice size.
import torch
p = torch.cuda.get_device_properties(0)
gb = p.total_memory / 1e9
print(f"{p.name}  {gb:.1f} GB  cc {p.major}.{p.minor}  SMs {p.multi_processor_count}")
if "A100" not in p.name:
    print("*** Not an A100. Runtime -> Change runtime type -> A100 GPU.")
print(f"\nDevice capacity is {gb:.0f} GB, but the BUDGET is ~10-12 GB per team.")
print("Those are different numbers; the budget is the binding one.")

## 6 - Data, transplant and controls`--prep-only` runs everything up to (but not including) fine-tuning: splits,scrambled Turkish, truncation profile, all six transplant artifacts, the C1identity/BPC controls, top-1 masked accuracy, embedding-norm report and thecross-base transplant-quality comparison. It then stops.

In [ ]:
!CONFIG=configs/experiment.yaml DEVICE=cuda bash run_all.sh --prep-only

In [ ]:
import json
s = json.load(open("results/splits.json", encoding="utf-8"))
print("splits:", s["az"]["train"], "/", s["az"]["val"], "/", s["az"]["test"],
      "| leakage:", s["leakage_check"])
assert not any(s["leakage_check"].values()), "TEST-SET LEAKAGE"

q = json.load(open("results/cross_base_transplant_quality.json", encoding="utf-8"))
for base, r in q["per_base"].items():
    t1t = r.get("top1_transplanted_canonical", {})
    print(f"{base:6s} dBPC={r.get('C1c_delta_bpc'):>9} norm_ratio={r.get('norm_ratio')} "
          f"healthy={r.get('norm_ratio_healthy')} top-1(transplanted)="
          f"{t1t.get('correct')}/{t1t.get('masked')}")
print("\nA transplant with ~0 top-1 on ~1,300 masked positions has lost masked-LM")
print("ability. Report that; and note it weakens the xlmr zero-effect control,")
print("since a null there cannot separate 'no deficit' from 'broken transplant'.")

## 7 - Micro-benchmark: measure s/step and per-process VRAMTwo 30-step benchmarks (Azerbaijani and Turkish stages), validation disabled,nothing written to `results/runs/`. This is **not** a training run.It prints the `run.vram_per_stream_gb` and `run.parallel_streams` values towrite back into the config, computed against the **team allocation** ratherthan the card.

In [ ]:
!bash scripts/a100_benchmark.sh

In [ ]:
import json, pathlib
p = pathlib.Path("results/hardware_benchmark.json")
if p.exists():
    b = json.load(open(p, encoding="utf-8"))
    print(f"AZ {b['az_stage']['mean_step_sec']:.3f} s/step | "
          f"TR {b['tr_stage']['mean_step_sec']:.3f} s/step")
    print(f"per-process peak {b['per_process_peak_gb']:.2f} GB "
          f"vs ceiling {b['vram_ceiling_gb']:.1f} GB")
    print(f"streams that fit: {b['streams_that_fit_ceiling']}  "
          f"-> recommended: {b['recommended_parallel_streams']}")
    print(f"\nserial full-grid projection: {b['full_grid_serial_sec']/3600:.1f} h")
    STREAMS = b["recommended_parallel_streams"]
else:
    STREAMS = 1
print("\nSTREAMS =", STREAMS)

In [ ]:
# Write the measured VRAM figure back so the launcher's guard uses a real
# number rather than the conservative placeholder. Config values are
# hash-locked, so re-lock after editing -- `run.*` keys are execution
# machinery, not pre-registered science (configs/FROZEN.md).
import hashlib, json, re
from pathlib import Path

b = json.load(open("results/hardware_benchmark.json", encoding="utf-8"))
cfgp = Path("configs/experiment.yaml")
text = cfgp.read_text(encoding="utf-8")
text = re.sub(r"(  vram_per_stream_gb: )[\d.]+",
              rf"\g<1>{b['recommended_vram_per_stream_gb']:.2f}", text)
text = re.sub(r"(  parallel_streams: )\d+",
              rf"\g<1>{b['recommended_parallel_streams']}", text)
cfgp.write_text(text, encoding="utf-8")

lockp = Path("configs/CONFIG_HASHES.lock")
lock = json.loads(lockp.read_text(encoding="utf-8"))
lock["configs/experiment.yaml"] = hashlib.sha256(
    cfgp.read_bytes().replace(b"\r\n", b"\n")).hexdigest()
lockp.write_text(json.dumps(lock, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

!python -c "from src.utils import verify_config_hashes; verify_config_hashes(); print('re-locked OK')"

## 8 - Tranche A (~3 h)Ten runs: `xlm15`, n=2000, `baza` vs `tokenizator`, 5 seeds. **No Turkishstage**, so no Phase 1 is needed - which is exactly why this tranche can runfirst and cheaply. It is also the end-to-end integration test: result writing,escape detection and post-hoc selection running together on real data.Watch the first two runs rather than walking away.

In [ ]:
!python -m src.training.run_grid --config configs/experiment.yaml --tranche A --phase 2 --streams 1

## 9 - The decision pointFixed before the data (run plan section 8) so the outcome cannot bereinterpreted afterwards. **Stop here and read the output** before runningPhase 1.

In [ ]:
!python -m src.analysis.aggregate --config configs/experiment.yaml
!python -m src.analysis.decompose --config configs/experiment.yaml

import json
cells = {(c["base"], c["condition"]): c
         for c in json.load(open("results/decompose.json", encoding="utf-8"))["cells"]}
baza, omp = cells[("xlm15", "baza")], cells[("xlm15", "tokenizator")]
print(f"baza        escape {baza['escape_rate_count']}  conditional F1 {baza['conditional_macro_f1']}")
print(f"tokenizator escape {omp['escape_rate_count']}  conditional F1 {omp['conditional_macro_f1']}")
b, o = baza["n_escaped"], omp["n_escaped"]
print()
if b == 0 and o == 0:
    print("NEITHER ESCAPES -> STOP AND RE-PLAN at n=10000. Do NOT tune the")
    print("optimizer, and do NOT start Phase 1. This is a finding about")
    print("detectability, not a bug.")
elif o >= 3 and b < 3:
    print("tokenizator ESCAPES, baza DOES NOT -> strong M3 evidence. Headline")
    print("result; proceed to Tranche B, then Phase 1.")
elif b >= 3 and o >= 3:
    print("BOTH ESCAPE -> pipeline works. Proceed to B, then Phase 1.")
else:
    print("ERRATIC -> seed variance may dominate. Report escape rate as the")
    print("primary outcome; raise seeds at the reference size only.")

In [ ]:
# Tranche B - the zero-effect control base. A and B together give the complete
# cross-base tokenizer figure, a reportable result before any Turkish run exists.
!python -m src.training.run_grid --config configs/experiment.yaml --tranche B --phase 2 --streams 1

## 10 - Phase 1: build and seal the Turkish cache30 checkpoints, ~33,780 steps, **serial by design** - this is the phaseboundary that makes parallel execution safe, so it is not parallelised eventhough it is the single longest sequential block (~4-6 h). Nothing scientificis measured here.

In [ ]:
!python -m src.training.run_grid --config configs/experiment.yaml --phase 1

In [ ]:
import json
m = json.load(open("artifacts/tr_stage_cache/phase1_manifest.json", encoding="utf-8"))
print("sealed:", m["sealed"], "| checkpoints:", m["n_distinct_checkpoints"])
built = sum(1 for e in m["entries"] if e.get("status") == "BUILT")
print("built this run:", built, "| reused from cache:", m["n_distinct_checkpoints"] - built)

## 11 - Phase 2: the rest of the gridThe cache is now read-only: a miss is a loud refusal, not a rebuild. Each queueitem is claimed atomically on dequeue, streams are separate OS processes (so noglobal RNG state is shared), and every result file records `streams_active`.> A per-run wall-clock measured under contention is **not** that run's isolated> cost. Never report it as a timing figure - use the step time from the> single-stream benchmark instead.

In [ ]:
# STREAMS came from the benchmark cell; the launcher caps it to the VRAM
# ceiling regardless, and logs any reduction.
!python -m src.training.run_grid --config configs/experiment.yaml --phase 2 --streams {STREAMS}

In [ ]:
import json
st = json.load(open("results/launcher_state_phase2.json", encoding="utf-8"))
print(f"completed {len(st['completed'])} | skipped {len(st['skipped_existing'])} "
      f"| failed {len(st['failures'])} | {st['elapsed_sec']/3600:.2f} h "
      f"| streams {st['streams_active']}")
for f in st["failures"]: print("  FAILED:", f)
print("\n" + st["timing_caveat"])

## 12 - Analysis

In [ ]:
# ---------------------------------------------------------------- analysis
# Every table and figure is regenerated from results/ alone; nothing is read
# from a notebook variable. Empty cells print NOT MEASURED rather than being
# silently dropped.
!python -m src.analysis.aggregate --config configs/experiment.yaml
!python -m src.analysis.stats     --config configs/experiment.yaml
!python -m src.analysis.decompose --config configs/experiment.yaml
!python -m src.analysis.report    --config configs/experiment.yaml

from IPython.display import Markdown, display
display(Markdown(open("results/paper_tables.md", encoding="utf-8").read()))

In [ ]:
from IPython.display import Image, display
for name in ("escape_rate.png", "conditional_macro_f1.png"):
    p = f"figures/{name}"
    print(p); display(Image(p))

In [ ]:
import json
d = json.load(open("results/decompose.json", encoding="utf-8"))
cross = d["cross_base_comparison"]
print("policy:", d["metric_policy"], "\n")
if cross.get("status") != "MEASURED":
    print("cross-base comparison: NOT MEASURED -", cross.get("missing"))
else:
    for base, r in cross["bases"].items():
        print(f"{base}: escape {r['baseline_escape_rate']} -> {r['omp_escape_rate']} "
              f"(diff {r['escape_rate_difference']:+.3f}); conditional F1 diff "
              f"{r['conditional_macro_f1_difference']}")
    c = cross["cross_base"]
    print(f"\nescape-rate effect difference:      {c['escape_rate_effect_difference']}")
    print(f"conditional-F1 effect difference:   {c['conditional_macro_f1_effect_difference']}")
    print("\nThe M3 claim requires the effect to be PRESENT in xlm15 (no Azerbaijani")
    print("in pre-training) and ABSENT in xlmr (has it). Check the control is")
    print("healthy before reading the treatment: a broken control manufactures a")
    print("false confirmation.")

In [ ]:
# Final archive.
import shutil
shutil.make_archive("/content/results_full", "zip", "results")
shutil.make_archive("/content/figures_full", "zip", "figures")
print("results_full.zip / figures_full.zip ready in /content")

## Test-set protocolEverything above evaluates on **validation**. The test set is touched exactlyonce, for the final numbers, through an explicit opt-in that appends to`results/test_evaluation_ledger.jsonl`:```python -m src.training.finetune --config configs/experiment.yaml \    --base primary --condition tokenizator --train-size 2000 --seed 42 \    --eval-split test --allow-test-eval```